In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rijulshr/pneumoniamnist/pneumoniamnist.npz


In [10]:
import numpy as np, os, glob
path = glob.glob('/kaggle/input/**/pneumoniamnist*.npz', recursive=True)[0]
print(path)
d = np.load(path)

os.makedirs('data', exist_ok=True)
for split in ['train', 'test']:
    X = d[f'{split}_images'].reshape(-1, 784).astype(np.float32) / 255.0
    y = d[f'{split}_labels'].reshape(-1).astype(np.uint8)
    print(split, X.shape, np.bincount(y))
    X.tofile(f'data/{split}_x.bin')
    y.tofile(f'data/{split}_y.bin')

/kaggle/input/datasets/rijulshr/pneumoniamnist/pneumoniamnist.npz
train (3882, 784) [ 388 3494]
test (624, 784) [234 390]


In [11]:
%%writefile fedml_compact.cu
#include <cstdio>
#include <cstdlib>
#include <cstdint>
#include <cstring>
#include <cmath>
#include <string>
#include <vector>
#include <numeric>
#include <algorithm>
#include <random>
#include <curand_kernel.h>

#define CK(x) do { cudaError_t e_ = (x); if (e_ != cudaSuccess) { fprintf(stderr, "CUDA error: %s (line %d)\n", cudaGetErrorString(e_), __LINE__); exit(1); } } while (0)

constexpr int D = 784, P = D + 1, KMAX = 16, TPB = 256;
enum Attack { NONE, LABEL_FLIP, MODEL_POISON };
enum Agg { MEAN, MEDIAN, TRIMMED };
static unsigned long long g_step = 0;

template <class T> T* dalloc(size_t n) { T* p; CK(cudaMalloc(&p, n * sizeof(T))); return p; }

__global__ void residual_kernel(const float* X, const uint8_t* y, const float* w, const int* idx, int B, float* r, float* xnorm) {
    int b = blockIdx.x;
    if (b >= B) return;
    int row = idx[b];
    const float* x = X + (size_t)row * D;
    float dot = 0.f, n2 = 0.f;
    for (int j = threadIdx.x; j < D; j += blockDim.x) { dot += w[j] * x[j]; n2 += x[j] * x[j]; }
    __shared__ float sd[TPB], sn[TPB];
    sd[threadIdx.x] = dot; sn[threadIdx.x] = n2;
    __syncthreads();
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {
        if (threadIdx.x < s) { sd[threadIdx.x] += sd[threadIdx.x + s]; sn[threadIdx.x] += sn[threadIdx.x + s]; }
        __syncthreads();
    }
    if (threadIdx.x == 0) {
        float p = 1.f / (1.f + expf(-(sd[0] + w[D])));
        r[b] = p - (float)y[row];
        xnorm[b] = sqrtf(sn[0] + 1.f);
    }
}

__global__ void grad_kernel(const float* X, const float* r, const float* xnorm, const int* idx, int B, float clip, float* g) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= P) return;
    float acc = 0.f;
    for (int b = 0; b < B; ++b) {
        float c = clip > 0.f ? fminf(1.f, clip / (fabsf(r[b]) * xnorm[b] + 1e-12f)) : 1.f;
        float xv = (j < D) ? X[(size_t)idx[b] * D + j] : 1.f;
        acc += c * r[b] * xv;
    }
    g[j] = acc;
}

__global__ void sgd_kernel(float* w, const float* g, int B, float lr, float noise_std, unsigned long long seed) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= P) return;
    float n = 0.f;
    if (noise_std > 0.f) {
        curandStatePhilox4_32_10_t st;
        curand_init(seed, j, 0, &st);
        n = noise_std * curand_normal(&st);
    }
    w[j] -= lr * (g[j] + n) / (float)B;
}

__global__ void sub_kernel(float* out, const float* a, const float* b, int n) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j < n) out[j] = a[j] - b[j];
}

__global__ void scale_kernel(float* a, float s, int n) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j < n) a[j] *= s;
}

__global__ void aggregate_kernel(const float* deltas, int K, int mode, int trim, float* w) {
    int j = blockIdx.x * blockDim.x + threadIdx.x;
    if (j >= P) return;
    float v[KMAX], out = 0.f;
    for (int k = 0; k < K; ++k) v[k] = deltas[(size_t)k * P + j];
    if (mode == MEAN) {
        for (int k = 0; k < K; ++k) out += v[k];
        out /= (float)K;
    } else {
        for (int i = 1; i < K; ++i) {
            float key = v[i];
            int m = i - 1;
            while (m >= 0 && v[m] > key) { v[m + 1] = v[m]; --m; }
            v[m + 1] = key;
        }
        if (mode == MEDIAN) {
            out = (K % 2) ? v[K / 2] : 0.5f * (v[K / 2 - 1] + v[K / 2]);
        } else {
            for (int i = trim; i < K - trim; ++i) out += v[i];
            out /= (float)(K - 2 * trim);
        }
    }
    w[j] += out;
}

__global__ void eval_kernel(const float* X, const uint8_t* y, const float* w, int N, int* correct) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    float z = w[D];
    for (int j = 0; j < D; ++j) z += w[j] * X[(size_t)i * D + j];
    if ((z > 0.f ? 1 : 0) == (int)y[i]) atomicAdd(correct, 1);
}

struct Dataset { std::vector<float> X; std::vector<uint8_t> y; int n = 0; };
struct DevData { float* X = nullptr; uint8_t* y = nullptr; int n = 0; };

template <class T> static bool load_vec(const std::string& path, std::vector<T>& out) {
    FILE* f = fopen(path.c_str(), "rb");
    if (!f) return false;
    fseek(f, 0, SEEK_END);
    long sz = ftell(f);
    fseek(f, 0, SEEK_SET);
    out.resize(sz / sizeof(T));
    size_t got = fread(out.data(), sizeof(T), out.size(), f);
    fclose(f);
    return got == out.size();
}

static bool load_dataset(const std::string& dir, const std::string& split, Dataset& d) {
    if (!load_vec(dir + "/" + split + "_x.bin", d.X) || !load_vec(dir + "/" + split + "_y.bin", d.y)) return false;
    d.n = (int)d.y.size();
    return (size_t)d.n * D == d.X.size();
}

static DevData upload(const Dataset& d, const std::vector<int>& rows, bool flip) {
    DevData dd;
    dd.n = (int)rows.size();
    std::vector<float> hx((size_t)dd.n * D);
    std::vector<uint8_t> hy(dd.n);
    for (int i = 0; i < dd.n; ++i) {
        memcpy(&hx[(size_t)i * D], &d.X[(size_t)rows[i] * D], D * sizeof(float));
        hy[i] = flip ? (uint8_t)(1 - d.y[rows[i]]) : d.y[rows[i]];
    }
    dd.X = dalloc<float>(hx.size());
    dd.y = dalloc<uint8_t>(hy.size());
    CK(cudaMemcpy(dd.X, hx.data(), hx.size() * sizeof(float), cudaMemcpyHostToDevice));
    CK(cudaMemcpy(dd.y, hy.data(), hy.size(), cudaMemcpyHostToDevice));
    return dd;
}

struct Cfg {
    const char* name = "";
    int clients = 5, rounds = 30, local_epochs = 1, batch = 64;
    float lr = 0.1f;
    bool dp = false;
    float clip = 1.0f, sigma = 1.0f;
    int attack = NONE, n_malicious = 1;
    float boost = 5.0f;
    int agg = MEAN, trim = 1;
};

static float evaluate(const DevData& te, const float* d_w, int* d_correct) {
    CK(cudaMemset(d_correct, 0, sizeof(int)));
    eval_kernel<<<(te.n + TPB - 1) / TPB, TPB>>>(te.X, te.y, d_w, te.n, d_correct);
    int h = 0;
    CK(cudaMemcpy(&h, d_correct, sizeof(int), cudaMemcpyDeviceToHost));
    return (float)h / (float)te.n;
}

static std::vector<float> run_experiment(const Cfg& c, const Dataset& train, const DevData& test) {
    std::mt19937 rng(42);
    int K = c.clients, gridP = (P + TPB - 1) / TPB;
    std::vector<int> perm(train.n);
    std::iota(perm.begin(), perm.end(), 0);
    std::shuffle(perm.begin(), perm.end(), rng);
    int per = train.n / K;
    std::vector<DevData> cd(K);
    for (int k = 0; k < K; ++k) {
        std::vector<int> rows(perm.begin() + (size_t)k * per, perm.begin() + (size_t)(k + 1) * per);
        cd[k] = upload(train, rows, c.attack == LABEL_FLIP && k < c.n_malicious);
    }
    float *d_w = dalloc<float>(P), *d_local = dalloc<float>(P), *d_g = dalloc<float>(P);
    float *d_deltas = dalloc<float>((size_t)K * P), *d_r = dalloc<float>(c.batch), *d_xn = dalloc<float>(c.batch);
    int *d_idx = dalloc<int>(c.batch), *d_correct = dalloc<int>(1);
    CK(cudaMemset(d_w, 0, P * sizeof(float)));
    std::vector<float> acc;
    for (int round = 0; round < c.rounds; ++round) {
        for (int k = 0; k < K; ++k) {
            CK(cudaMemcpy(d_local, d_w, P * sizeof(float), cudaMemcpyDeviceToDevice));
            std::vector<int> order(cd[k].n);
            std::iota(order.begin(), order.end(), 0);
            for (int e = 0; e < c.local_epochs; ++e) {
                std::shuffle(order.begin(), order.end(), rng);
                for (int s = 0; s < cd[k].n; s += c.batch) {
                    int B = std::min(c.batch, cd[k].n - s);
                    CK(cudaMemcpy(d_idx, &order[s], B * sizeof(int), cudaMemcpyHostToDevice));
                    residual_kernel<<<B, TPB>>>(cd[k].X, cd[k].y, d_local, d_idx, B, d_r, d_xn);
                    grad_kernel<<<gridP, TPB>>>(cd[k].X, d_r, d_xn, d_idx, B, c.dp ? c.clip : 0.f, d_g);
                    sgd_kernel<<<gridP, TPB>>>(d_local, d_g, B, c.lr, c.dp ? c.sigma * c.clip : 0.f, 1234ull + 7919ull * (++g_step));
                }
            }
            sub_kernel<<<gridP, TPB>>>(d_deltas + (size_t)k * P, d_local, d_w, P);
            if (c.attack == MODEL_POISON && k < c.n_malicious) scale_kernel<<<gridP, TPB>>>(d_deltas + (size_t)k * P, -c.boost, P);
        }
        aggregate_kernel<<<gridP, TPB>>>(d_deltas, K, c.agg, c.trim, d_w);
        CK(cudaGetLastError());
        acc.push_back(evaluate(test, d_w, d_correct));
    }
    for (auto& d : cd) { cudaFree(d.X); cudaFree(d.y); }
    for (void* p : {(void*)d_w, (void*)d_local, (void*)d_g, (void*)d_deltas, (void*)d_r, (void*)d_xn, (void*)d_idx, (void*)d_correct}) cudaFree(p);
    return acc;
}

int main(int argc, char** argv) {
    cudaDeviceProp prop;
    CK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s\n", prop.name);
    if (argc < 2) { fprintf(stderr, "Usage: %s <data_dir>\n", argv[0]); return 1; }
    Dataset train, test;
    if (!load_dataset(argv[1], "train", train) || !load_dataset(argv[1], "test", test)) { fprintf(stderr, "Could not load data from %s\n", argv[1]); return 1; }
    printf("Data: %d train, %d test\n", train.n, test.n);
    std::vector<int> all(test.n);
    std::iota(all.begin(), all.end(), 0);
    DevData dtest = upload(test, all, false);

    std::vector<Cfg> exps;
    auto add = [&](const char* name, auto mod) { Cfg c; c.name = name; mod(c); exps.push_back(c); };
    add("central (no FL)",           [](Cfg& c) { c.clients = 1; });
    add("FedAvg",                    [](Cfg& c) {});
    add("FedAvg + DP",               [](Cfg& c) { c.dp = true; c.lr = 1.0f; });
    add("label flip, mean",          [](Cfg& c) { c.attack = LABEL_FLIP; c.agg = MEAN; });
    add("label flip, median",        [](Cfg& c) { c.attack = LABEL_FLIP; c.agg = MEDIAN; });
    add("label flip, trimmed",       [](Cfg& c) { c.attack = LABEL_FLIP; c.agg = TRIMMED; });
    add("model poison, mean",        [](Cfg& c) { c.attack = MODEL_POISON; c.agg = MEAN; });
    add("model poison, median",      [](Cfg& c) { c.attack = MODEL_POISON; c.agg = MEDIAN; });
    add("model poison, trimmed",     [](Cfg& c) { c.attack = MODEL_POISON; c.agg = TRIMMED; });
    add("DP + model poison, median", [](Cfg& c) { c.dp = true; c.lr = 1.0f; c.attack = MODEL_POISON; c.agg = MEDIAN; });

    FILE* csv = fopen("results.csv", "w");
    if (csv) fprintf(csv, "experiment,round,test_accuracy\n");
    printf("\n%-28s %s\n", "experiment", "final test acc");
    for (const Cfg& c : exps) {
        std::vector<float> acc = run_experiment(c, train, dtest);
        printf("%-28s %.4f\n", c.name, acc.back());
        if (csv) for (size_t r = 0; r < acc.size(); ++r) fprintf(csv, "%s,%zu,%.5f\n", c.name, r + 1, acc[r]);
    }
    if (csv) fclose(csv);
    printf("\nPer-round accuracy written to results.csv\n");
    cudaFree(dtest.X); cudaFree(dtest.y);
    return 0;
}

Overwriting fedml_compact.cu


In [12]:
!nvcc -O2 -std=c++17 fedml_compact.cu -o fedml_compact

In [13]:
!./fedml_compact /kaggle/working/data

GPU: Tesla T4
Data: 3882 train, 624 test

experiment                   final test acc
central (no FL)              0.7981
FedAvg                       0.6971
FedAvg + DP                  0.6603
label flip, mean             0.7933
label flip, median           0.7724
label flip, trimmed          0.7468
model poison, mean           0.3750
model poison, median         0.7692
model poison, trimmed        0.7548
DP + model poison, median    0.7324

Per-round accuracy written to results.csv
